# Preliminari

Si impostano directory di lavoro e si fanno import per spark

In [7]:
import os
from pyspark.sql import SparkSession

DATASETS_DIR = "../dataset/"

spark = (
    SparkSession.builder
    .appName("pfp")
    .getOrCreate()
)

sc = spark.sparkContext


## Pre-Processing


Creo un dataframe dal file .parquet di input.

Creo i record (rdd) come necessario dal problema, ovvero chiave dell'ordine e valore le tuple contenente id oggetto e quantità.
Questi record li chiameremo Transazioni, come suggerito dal paper PFP.

In [8]:
sdf = spark.read.parquet(
    os.path.join(DATASETS_DIR, "online_retail_II.parquet")
)

transactions = (
    sdf
    .select("Invoice", "StockCode", "Quantity")
    .rdd
    .map(lambda row: (row["Invoice"], (row["StockCode"], int(row["Quantity"]))))
    .groupByKey()
    .mapValues(list)
)
transactions.take(5)

[('489434',
  [('85048', 12),
   ('79323P', 12),
   ('79323W', 12),
   ('22041', 48),
   ('21232', 24),
   ('22064', 24),
   ('21871', 24),
   ('21523', 10)]),
 ('489435', [('22350', 12), ('22349', 12), ('22195', 24), ('22353', 12)]),
 ('489436',
  [('48173C', 10),
   ('21755', 18),
   ('21754', 3),
   ('84879', 16),
   ('22119', 3),
   ('22142', 12),
   ('22296', 12),
   ('22295', 12),
   ('22109', 16),
   ('22107', 4),
   ('22194', 2),
   ('35004B', 12),
   ('82582', 12),
   ('21181', 12),
   ('21756', 3),
   ('21333', 6),
   ('84596F', 8),
   ('84596L', 8),
   ('22111', 24)]),
 ('489437',
  [('22143', 6),
   ('22145', 6),
   ('22130', 12),
   ('21364', 2),
   ('21360', 1),
   ('21351', 2),
   ('21352', 2),
   ('35400', 2),
   ('20695', 3),
   ('37370', 12),
   ('10002', 12),
   ('84507B', 6),
   ('20703', 3),
   ('21987', 12),
   ('21989', 12),
   ('84970S', 12),
   ('20971', 12),
   ('22271', 6),
   ('22272', 6),
   ('22274', 6),
   ('21912', 4),
   ('22111', 3),
   ('22112', 3)]),

### Conversione

convertiamo gli oggetti (tuple chiave e quantità) in nuovi oggetti identificati da un numero, in questo modo si può facilmente utilizzare PFP con le quantità.
Riduciamo funzionalmente il problema di tenere in considerazione le quantità al problema senza le quantità per poi tornare al problema delle quantità.

T' = T
for t in T'
    t -> t'

out = PFP(T')

reversed = revert(out)

return alpha_code(reversed)

Per fare tutto ciò innanzitutto devo prendere gli oggetti e quantità e mapparli:

In [9]:
pairs= (
    transactions
    .flatMap(lambda x: x[1])                 # prendo tutte le tuple
    .distinct()                              # tuple uniche
    .sortBy(lambda pair: (pair[0], pair[1])) # ordine stabile
    .zipWithIndex()                          # assegna indice 0,1,2...
)

print("ci sono " + str(pairs.count()) + " coppie")
pairs.take(50)


ci sono 45389 coppie


[(('10002', -400), 0),
 (('10002', -200), 1),
 (('10002', -12), 2),
 (('10002', -10), 3),
 (('10002', -1), 4),
 (('10002', 1), 5),
 (('10002', 2), 6),
 (('10002', 3), 7),
 (('10002', 4), 8),
 (('10002', 5), 9),
 (('10002', 6), 10),
 (('10002', 7), 11),
 (('10002', 8), 12),
 (('10002', 9), 13),
 (('10002', 10), 14),
 (('10002', 12), 15),
 (('10002', 13), 16),
 (('10002', 18), 17),
 (('10002', 20), 18),
 (('10002', 21), 19),
 (('10002', 22), 20),
 (('10002', 24), 21),
 (('10002', 26), 22),
 (('10002', 30), 23),
 (('10002', 36), 24),
 (('10002', 40), 25),
 (('10002', 42), 26),
 (('10002', 48), 27),
 (('10002', 60), 28),
 (('10002', 96), 29),
 (('10002', 100), 30),
 (('10002', 110), 31),
 (('10002', 120), 32),
 (('10002', 200), 33),
 (('10002', 250), 34),
 (('10002', 300), 35),
 (('10002', 400), 36),
 (('10002R', 1), 37),
 (('10002R', 2), 38),
 (('10080', 1), 39),
 (('10080', 2), 40),
 (('10080', 6), 41),
 (('10080', 90), 42),
 (('10109', -4), 43),
 (('10109', 4), 44),
 (('10120', -9000), 

In [10]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)
bc_map.value

{('10002', -400): 0,
 ('10002', -200): 1,
 ('10002', -12): 2,
 ('10002', -10): 3,
 ('10002', -1): 4,
 ('10002', 1): 5,
 ('10002', 2): 6,
 ('10002', 3): 7,
 ('10002', 4): 8,
 ('10002', 5): 9,
 ('10002', 6): 10,
 ('10002', 7): 11,
 ('10002', 8): 12,
 ('10002', 9): 13,
 ('10002', 10): 14,
 ('10002', 12): 15,
 ('10002', 13): 16,
 ('10002', 18): 17,
 ('10002', 20): 18,
 ('10002', 21): 19,
 ('10002', 22): 20,
 ('10002', 24): 21,
 ('10002', 26): 22,
 ('10002', 30): 23,
 ('10002', 36): 24,
 ('10002', 40): 25,
 ('10002', 42): 26,
 ('10002', 48): 27,
 ('10002', 60): 28,
 ('10002', 96): 29,
 ('10002', 100): 30,
 ('10002', 110): 31,
 ('10002', 120): 32,
 ('10002', 200): 33,
 ('10002', 250): 34,
 ('10002', 300): 35,
 ('10002', 400): 36,
 ('10002R', 1): 37,
 ('10002R', 2): 38,
 ('10080', 1): 39,
 ('10080', 2): 40,
 ('10080', 6): 41,
 ('10080', 90): 42,
 ('10109', -4): 43,
 ('10109', 4): 44,
 ('10120', -9000): 45,
 ('10120', -30): 46,
 ('10120', 1): 47,
 ('10120', 2): 48,
 ('10120', 3): 49,
 ('10120'

In [11]:
# Rimappo ogni transazione
transactions_ids = transactions.mapValues(
    lambda items: [bc_map.value[item] for item in items]
)
print(transactions_ids.count())
transactions_ids.take(5)

28816


[('489434', [41095, 34438, 34500, 16083, 7555, 16359, 14253, 10346]),
 ('489435', [20826, 20811, 18576, 20871]),
 ('489436',
  [32052,
   13015,
   12984,
   38971,
   17261,
   17616,
   19972,
   19950,
   17078,
   17048,
   18553,
   28907,
   34992,
   6855,
   13025,
   8568,
   37152,
   37197,
   17123]),
 ('489437',
  [17630,
   17666,
   17397,
   8849,
   8815,
   8699,
   8712,
   29084,
   2286,
   30125,
   15,
   36445,
   2346,
   15742,
   15787,
   39873,
   4535,
   19618,
   19637,
   19675,
   14832,
   17109,
   17135]),
 ('489438',
  [8524,
   7799,
   5961,
   5217,
   2409,
   9264,
   9277,
   35400,
   35412,
   35436,
   35461,
   36601,
   36614,
   42351,
   42369,
   42867,
   42882])]

## PFP

Iniziamo ad implementare **PFP**, definiamo una variabile **epsilon** che rappresenta la "predefined minimum support threshold"
soglia minima predefinita di supporto.
Quindi una threshold sopra la quale verrà riconosciuto un pattern e i pattern sotto questa soglia verranno scartati 

In [12]:
epsilon = 10

#supporto(item) = numero di transazioni che contengono lo stesso item

item_counts = (
    transactions_ids
    .flatMap(lambda row: set(row[1]))   # ogni item contato una sola volta per transazione
    .map(lambda item: (item, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda row: row[1] >= epsilon)
)
print(item_counts.count())
item_counts.take(10)


10169


[(7555, 134),
 (41095, 38),
 (16359, 18),
 (10346, 80),
 (14253, 25),
 (16083, 32),
 (18576, 11),
 (20826, 12),
 (20811, 12),
 (20871, 29)]

Creo la F-List che è la lista decrescente degli item (item = id_of(tuple(code, quantity)))

Poi ordino le transazione per "supporto" ovvero in base ai valori di F-List

Creo infine la Q-List tramite la quale si suddivide il calcolo tra le macchine.

In [13]:
# creo f_list
f_list = item_counts.sortBy(
    lambda row: (row[1], row[0]),
    ascending=False
)

f_list.take(10)

[(42083, 818),
 (7236, 811),
 (45272, 733),
 (41693, 662),
 (2683, 646),
 (5224, 625),
 (21878, 609),
 (40207, 569),
 (38964, 552),
 (7545, 526)]

In [14]:
# Creo una mappa per ordinare le transazioni, la mappa è fatta così:  item_id -> posizione nella F-list

# Base comune: item_id -> rank nella F-list
item_rank = (
    f_list
    .map(lambda row: row[0])      # item_id
    .zipWithIndex()               # item_id -> rank
    .map(lambda x: (x[0], int(x[1])))
    .persist()
)

f_rank = item_rank.collectAsMap()
# notifico i worker
bc_f_rank = sc.broadcast(f_rank)

ordered_transactions = (
    transactions_ids
    .mapValues(
        lambda items: sorted(
            # tieni item solo se item è una chiave del dizionario f_rank
            set(item for item in items if item in bc_f_rank.value),
            key=lambda item: bc_f_rank.value[item]
        )
    )
    .filter(lambda row: len(row[1]) > 0)
)
print(ordered_transactions.count())
print(ordered_transactions.take(5))


21850
[('489434', [7555, 10346, 41095, 16083, 14253, 16359]), ('489435', [20871, 20826, 20811, 18576]), ('489436', [12984, 38971, 6855, 19972, 19950, 13025, 34992, 18553, 17261, 17123, 37152, 32052, 17048, 17616, 28907, 17078, 13015, 8568, 37197]), ('489437', [39873, 4535, 17109, 17135, 29084, 19618, 14832, 19675, 17397, 15, 30125, 2346, 15787, 19637, 15742, 17666, 8849, 8815, 8699, 36445, 8712, 2286, 17630]), ('489439', [40477, 3019, 720, 40460, 45332, 20548, 16357, 43334, 12698, 16382, 20849, 37768, 43614, 17399, 791, 17569, 17541, 10138])]


In [15]:
# G-List
Q = spark.sparkContext.defaultParallelism # numero di core disponibili tra tutti i worker

g_list = (
    item_rank
    .map(lambda x: (x[0], int(x[1] % Q)))   # item_id -> gid
    .collectAsMap()
)

bc_g_list = sc.broadcast(g_list)

In [16]:
# Questo è fondamentalmente il mapper del paper
def generate_group_dependent_transactions(row):
    invoice_no, items = row

    output = []
    seen_gids = set()
    
    # Scorro la transazione da destra verso sinistra
    for j in range(len(items) - 1, -1, -1):
        item = items[j]
        gid = bc_g_list.value.get(item)

        # Se questo gruppo non è ancora stato emesso per questa transazione
        if gid is not None and gid not in seen_gids:
            seen_gids.add(gid)

            # Emetto il prefisso fino alla posizione j inclusa
            output.append((gid, items[:j + 1]))

    return output

group_dependent_transactions = ordered_transactions.flatMap(
    generate_group_dependent_transactions
)

group_dependent_transactions.take(10)

# a ogni gid associo le transazioni di cui si deve occupare.

group_shards = group_dependent_transactions.groupByKey()

### Nodi e Alberi

A questo punto abbiamo bisogno degli FP-tree per minare i pattern. 
Ogni worker/reducer costruirà un FP-tree locale a partire dalle transazioni associate a uno specifico gruppo.

Dato che gli item di ogni transazione sono già ordinati secondo la F-list, e dato che le transazioni group-dependent sono raggruppate per `gid`, ogni gruppo può essere minato in modo indipendente dagli altri (come dimostrato nel paper).

Una transazione originale può generare più transazioni parziali, una per ogni gruppo presente nella transazione. Durante lo shuffle, questi prefissi vengono inviati ai reducer corrispondenti ai rispettivi gruppi.

Quindi una stessa transazione originale può contribuire a più FP-tree locali, ma ogni FP-tree locale contiene solo il sotto-database necessario per minare i pattern che terminano negli item del proprio gruppo.

In [17]:

from collections import defaultdict

# Creiamo una struttura di nodi in grado di navigare al parent e ai child.
class FPNode:
    def __init__(self, item=None, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

    def add_child(self, item):
        child = FPNode(item=item, parent=self)
        self.children[item] = child
        return child


# L'inserimento di una transazione può comportare la creazione di nodi figli oppure l'incremento del loro conteggio.

def insert_transaction(root, transaction, header_table, count=1):
    node = root
    for item in transaction:

        if item in node.children:
            child = node.children[item]
            child.count += count
        else:
            child = node.add_child(item)
            child.count = count
            header_table[item].append(child)

        node = child
    
# La header-table serve per avere una navigazione rapida ai nodi che rappresentano lo stesso item, infatti nell'albero possono 
# esserci N nodi che rappresentano lo stesso item, grazie alla header_table possiamo evitare di navigare tutto l'albero 
# ma abbiamo un accesso "diretto"

def build_fp_tree(transactions_iter):
    root = FPNode()
    header_table = defaultdict(list)

    for transaction in transactions_iter:
        insert_transaction(root, transaction, header_table, count=1)

    return root, dict(header_table) 

def is_single_path(node):
    current = node

    while True:
        if len(current.children) == 0:
            return True
        if len(current.children) > 1:
            return False

        current = next(iter(current.children.values()))

        
def print_tree(node, indent=0, max_depth=3):
    if indent >= max_depth:
        return

    for child in node.children.values():
        print("-" * indent + f"{child.item}:{child.count}")
        print_tree(child, indent + 1, max_depth)



def count_nodes(node):
    total = 1
    for child in node.children.values():
        total += count_nodes(child)
    return total



def tree_stats_for_group(row):
    gid, transactions_iter = row

    root, header_table = build_fp_tree(transactions_iter)

    return (
        gid,
        count_nodes(root),
        len(header_table),
        list(root.children.keys())[:10]
    )

In [18]:
# Ora creiamo la nowGroup

gid_to_items_tmp = defaultdict(list)

for item_id, gid in g_list.items():
    gid_to_items_tmp[gid].append(item_id)
# gid -> item_ids
nowGroup = dict(gid_to_items_tmp)

bc_nowGroup = sc.broadcast(nowGroup)

In [19]:
#tree_stats = group_shards.map(tree_stats_for_group)
#gid, nodes_number, header_table_length,childern_length = tree_stats.take(1)[0]
#tree_stats.take(10)
#print("gid:", gid)
#print("item distinti nell'albero:", header_table_length)

In [20]:
# Dato un nodo, risale fino alla root e restituisce il cammino dei parent.
# Esempio se il nodo è d ed il path è a -> b -> c -> d restituisce [a,b,c] 
# Quindi non torna root e nodo corrente

def get_prefix_path(node):
    path = []
    current = node.parent
    while current is not None and current.item is not None:
        path.append(current.item)
        current = current.parent
    path.reverse()
    return path


# Per un item, prende tutti i nodi in header_table[item] e costruisce una lista degli elementi precedenti a lui nel path


def conditional_pattern_base(item, header_table):
    base = []
    for node in header_table.get(item, []):
        path = get_prefix_path(node)
        if path:
            base.append((path, node.count))
    return base


# conta gli item
# fa pruning degli item sotto soglia
# ordina i prefissi
# costruisce un nuovo FP-tree condizionato

def build_conditional_tree(pattern_base, min_support):
    # 1. conta item nella base pesando con node.count
    item_counts = defaultdict(int)
    for path, count in pattern_base:
        for item in path:
            item_counts[item] += count

    # 2. pruning
    frequent_items = {item for item, c in item_counts.items() if c >= min_support}
    if not frequent_items:
        return None, {}

    # 3. costruisco transazioni filtrate e ordinate
    root = FPNode()
    header_table = defaultdict(list)

    for path, count in pattern_base:
        filtered = [item for item in path if item in frequent_items]
        if filtered:
            insert_transaction(root, filtered, header_table, count=count)

    return root, dict(header_table)


# Per ogni item nell’albero:
# crea il pattern prefix + item
# salva il supporto
# costruisce il conditional tree
# richiama ricorsivamente mine_fp_tree

def mine_fp_tree(root, header_table, min_support, suffix=()):
    patterns = []

    items = sorted(
        header_table.keys(),
        key=lambda item: sum(node.count for node in header_table[item])
    )

    for item in items:
        support = sum(node.count for node in header_table[item])

        if support < min_support:
            continue
            
        new_pattern = tuple(sorted((item,) + suffix))
        patterns.append((new_pattern, support))

        base = conditional_pattern_base(item, header_table)
        cond_root, cond_header = build_conditional_tree(base, min_support)

        if cond_header:
            patterns.extend(
                mine_fp_tree(cond_root, cond_header, min_support, suffix=(item,) + suffix)
            )

    return patterns

In [21]:
#TESTING CODE BOX
gid, transactions_iter = group_shards.take(1)[0]
root, header_table = build_fp_tree(transactions_iter)

patterns = mine_fp_tree(root, header_table, min_support=epsilon)
patterns[:20]

[((19201,), 10),
 ((44964,), 10),
 ((7860,), 10),
 ((40287,), 10),
 ((2078,), 10),
 ((34582,), 10),
 ((15391,), 10),
 ((11150,), 10),
 ((9969,), 10),
 ((16651,), 10),
 ((31784,), 10),
 ((6563,), 10),
 ((42258,), 10),
 ((33375,), 10),
 ((18051,), 10),
 ((8518,), 10),
 ((43655,), 10),
 ((43655, 45272), 10),
 ((8770,), 10),
 ((14611,), 10)]

In [22]:
def mine_group(row):
    gid, transactions_iter = row
    root, header_table = build_fp_tree(transactions_iter)
    return mine_fp_tree(root, header_table, min_support=epsilon)

all_patterns = group_shards.flatMap(mine_group)

In [23]:
all_patterns.take(1000)

# readable_patterns = all_patterns.map(
#     lambda x: f"pattern={x[0]} | support={x[1]}"
# )

# readable_patterns.saveAsTextFile("output_patterns")

[((19201,), 10),
 ((44964,), 10),
 ((7860,), 10),
 ((40287,), 10),
 ((2078,), 10),
 ((34582,), 10),
 ((15391,), 10),
 ((11150,), 10),
 ((9969,), 10),
 ((16651,), 10),
 ((31784,), 10),
 ((6563,), 10),
 ((42258,), 10),
 ((33375,), 10),
 ((18051,), 10),
 ((8518,), 10),
 ((43655,), 10),
 ((43655, 45272), 10),
 ((8770,), 10),
 ((14611,), 10),
 ((7692,), 10),
 ((9442,), 10),
 ((8244,), 10),
 ((31970,), 10),
 ((33443,), 10),
 ((34668,), 10),
 ((9667,), 10),
 ((4660,), 10),
 ((33567,), 10),
 ((2996,), 10),
 ((35183,), 10),
 ((17375,), 10),
 ((4171,), 10),
 ((3674,), 10),
 ((44883,), 10),
 ((44883, 45272), 10),
 ((18868,), 10),
 ((12068,), 10),
 ((1843,), 10),
 ((34081,), 10),
 ((15377,), 10),
 ((10026,), 10),
 ((11921,), 10),
 ((41966,), 10),
 ((17430,), 10),
 ((20757,), 10),
 ((20757, 45272), 10),
 ((13727,), 10),
 ((17905,), 10),
 ((17118,), 10),
 ((16722,), 10),
 ((5252,), 10),
 ((41395,), 10),
 ((7711,), 10),
 ((269,), 10),
 ((43537,), 10),
 ((45097,), 10),
 ((29740,), 10),
 ((17166,), 10)

In [24]:
invert_conversion_map = {v:k for k, v in conversion_map.items()}
bc_invert_conversion_map = sc.broadcast(invert_conversion_map)

def convert_to_qt(pattern):
    pattern, support = pattern
    return (tuple(bc_invert_conversion_map.value[i] for i in pattern), support)

real_patterns = all_patterns.map(convert_to_qt).persist()

In [25]:
from functools import reduce
from math import gcd
from pprint import pformat


def normalize_pattern(pattern):
    """
    pattern: ((StockCode, Quantity), ...)
    ritorna:
      shape: forma ridotta del pattern
      alpha: fattore di scala
      canonical_pattern: pattern ordinato
    """

    canonical_pattern = tuple(sorted(pattern, key=lambda x: x[0]))

    quantities = [abs(int(qty)) for item, qty in canonical_pattern]

    if not quantities:
        return None

    g = reduce(gcd, quantities)

    if g == 0:
        return None

    shape = tuple(
        (item, abs(int(qty)) // g)
        for item, qty in canonical_pattern
    )

    alpha = g

    return shape, alpha, canonical_pattern


def to_alpha_record(row):
    """
    row: (pattern, support)

    ritorna:
      (shape, (alpha, pattern, support))
    """

    pattern, support = row

    result = normalize_pattern(pattern)

    if result is None:
        return []

    shape, alpha, canonical_pattern = result

    return [
        (shape, (alpha, canonical_pattern, support))
    ]


def reduce_alpha_edges(alphas):
    """
    Data una lista di alpha, costruisce solo gli archi non ridondanti.

    Esempio:
      alpha = [1, 2, 3, 4]

    archi possibili:
      1 -> 2
      1 -> 3
      1 -> 4
      2 -> 4

    dopo riduzione:
      1 -> 2
      1 -> 3
      2 -> 4

    perché 1 -> 4 è ridondante.
    """

    alphas = sorted(set(alphas))
    edges = []

    for parent in alphas:
        for child in alphas:
            if parent >= child:
                continue

            if child % parent != 0:
                continue

            redundant = False

            for middle in alphas:
                if parent < middle < child:
                    if middle % parent == 0 and child % middle == 0:
                        redundant = True
                        break

            if not redundant:
                edges.append((parent, child, child // parent))

    return edges




alpha_records = (
    real_patterns
    .flatMap(to_alpha_record)
    .cache()
)

alpha_groups = (
    alpha_records
    .groupByKey()
    .mapValues(list)
    .cache()
)

def summarize_alpha_group(row):
    shape, values = row

    by_alpha = defaultdict(list)

    for alpha, pattern, support in values:
        by_alpha[alpha].append((pattern, support))

    alphas = sorted(by_alpha.keys())
    edges = reduce_alpha_edges(alphas)

    return {
        "shape": shape,
        "shape_size": len(shape),   # numero di oggetti nella shape
        "alpha_count": len(alphas),
        "pattern_count": sum(len(v) for v in by_alpha.values()),
        "alphas": alphas,
        "patterns_by_alpha": dict(by_alpha),
        "edges": edges,
        "edge_count": len(edges),
        "is_proportional": len(alphas) > 1,
    }


alpha_summary = (
    alpha_groups
    .map(summarize_alpha_group)
    .cache()
)
###
#alpha_summary_sorted = (
#    alpha_summary
#    .sortBy(lambda g: (-g["alpha_count"], -g["pattern_count"]))
#)
###

alpha_summary_sorted = (
    alpha_summary
    .filter(lambda g : g["is_proportional"])
    .sortBy(lambda g: (-g["shape_size"], -g["alpha_count"], -g["pattern_count"]))
)

#shape con più oggetti
#poi, a parità di numero di oggetti, quelle con più alpha
#poi, a parità di alpha, quelle con più pattern
alpha_summary_nontrivial_sorted = (
    alpha_summary
    .filter(lambda g: g["shape_size"] >= 2 and g["is_proportional"])
    .sortBy(lambda g: (-g["shape_size"], -g["alpha_count"], -g["pattern_count"]))
)

top_groups = alpha_summary_sorted.take(10)

alpha_summary = alpha_groups.map(summarize_alpha_group).cache()
alpha_summary_sorted = alpha_summary.sortBy(lambda g: (-g["alpha_count"], -g["pattern_count"]))
#alpha_summary_sorted = alpha_summary.sortBy(lambda g: (-g["pattern_count"], -g["alpha_count"]))

In [26]:
#readable / explainable
desc_map = (
    sdf
    .select("StockCode", "Description")
    .dropna()
    .rdd
    .map(lambda row: (str(row["StockCode"]), str(row["Description"])))
    .reduceByKey(lambda a, b: a)
    .collectAsMap()
)

bc_desc_map = sc.broadcast(desc_map)


def format_shape_readable(shape):
    """
    shape: (('22520', 1), ('22521', 1), ...)
    ritorna righe leggibili con descrizione prodotto
    """

    lines = []

    for item, qty in shape:
        description = bc_desc_map.value.get(str(item), "DESCRIPTION NOT FOUND")
        lines.append(f"  ({item}, {qty}) -> {description}")

    return "\n".join(lines)


# Gruppi proporzionali: 
proportional_groups = (
    alpha_summary
    #.filter(lambda g: g["shape_size"] >= 2)
    .filter(lambda g: g["is_proportional"])
    .cache()
)

# Gruppi non proporzionali: 
non_proportional_groups = (
    alpha_summary
    #.filter(lambda g: g["shape_size"] >= 2)
    .filter(lambda g: not g["is_proportional"])
    .cache()
)

proportional_sorted = (
    proportional_groups
    .sortBy(lambda g: (-g["shape_size"], -g["alpha_count"], -g["pattern_count"]))
)

non_proportional_sorted = (
    non_proportional_groups
    .sortBy(lambda g: (-g["shape_size"], -g["pattern_count"]))
)



with open("alpha_proportional_readable.txt", "w", encoding="utf-8") as f:
    for i, group in enumerate(proportional_sorted.toLocalIterator(), start=1):
        f.write(f"===== PROPORTIONAL GROUP {i} =====\n")
        f.write(f"NUMBER OF ITEMS IN SHAPE: {group['shape_size']}\n")
        f.write(f"ALPHA VALUES: {group['alphas']}\n")
        f.write(f"ALPHA COUNT: {group['alpha_count']}\n")
        f.write(f"PATTERN COUNT: {group['pattern_count']}\n\n")

        f.write("BASE SHAPE:\n")
        f.write(format_shape_readable(group["shape"]))
        f.write("\n\n")

        f.write("PROPORTIONAL PATTERNS:\n")
        for alpha in group["alphas"]:
            scaled_pattern = tuple(
                (item, qty * alpha)
                for item, qty in group["shape"]
            )

            f.write(f"  alpha = {alpha}\n")

            for item, qty in scaled_pattern:
                description = bc_desc_map.value.get(str(item), "DESCRIPTION NOT FOUND")
                f.write(f"    ({item}, {qty}) -> {description}\n")

            f.write("\n")

        f.write("REDUCED ALPHA EDGES:\n")
        f.write(str(group["edges"]))
        f.write("\n\n\n")

print("File scritto: alpha_proportional_readable.txt")



with open("alpha_non_proportional_readable.txt", "w", encoding="utf-8") as f:
    for i, group in enumerate(non_proportional_sorted.toLocalIterator(), start=1):
        f.write(f"===== NON PROPORTIONAL GROUP {i} =====\n")
        f.write(f"NUMBER OF ITEMS IN SHAPE: {group['shape_size']}\n")
        f.write(f"ALPHA VALUES: {group['alphas']}\n")
        f.write(f"PATTERN COUNT: {group['pattern_count']}\n\n")

        f.write("PATTERN:\n")
        f.write(format_shape_readable(group["shape"]))
        f.write("\n\n\n")

print("File scritto: alpha_non_proportional_readable.txt")


num_reduced_patterns = proportional_groups.count()
num_non_reduced_patterns = non_proportional_groups.count()
num_total_patterns_after_alpha = num_reduced_patterns + num_non_reduced_patterns

print("Pattern ridotti tramite alpha reduction:", num_reduced_patterns)
print("Pattern non ridotti:", num_non_reduced_patterns)
print("Totale pattern finali:", num_total_patterns_after_alpha)

File scritto: alpha_proportional_readable.txt


File scritto: alpha_non_proportional_readable.txt


Pattern ridotti tramite alpha reduction: 6836
Pattern non ridotti: 197059
Totale pattern finali: 203895


In [27]:
# tutta sta roba non serve più ma non si sa mai

top_groups = alpha_summary_nontrivial_sorted.take(10)

for i, group in enumerate(top_groups, start=1):
    print("==== GROUP", i, "====")
    print("shape_size:", group["shape_size"])
    print("alpha_count:", group["alpha_count"])
    print("pattern_count:", group["pattern_count"])
    print("edge_count:", group["edge_count"])
    print("is_proportional:", group["is_proportional"])
    print("shape:", group["shape"])
    print("alphas:", group["alphas"])
    print("edges:", group["edges"])
    print()

with open("alpha_groups_sorted_nontrivial.txt", "w", encoding="utf-8") as f:
    for i, group in enumerate(alpha_summary_nontrivial_sorted.toLocalIterator(), start=1):
        f.write(f"===== GROUP {i} =====\n")
        f.write(f"TYPE: {'PROPORTIONAL' if group['is_proportional'] else 'NON_PROPORTIONAL'}\n")
        f.write(f"SHAPE SIZE: {group['shape_size']}\n")
        f.write(f"ALPHA COUNT: {group['alpha_count']}\n")
        f.write(f"PATTERN COUNT: {group['pattern_count']}\n")
        f.write(f"EDGE COUNT: {group['edge_count']}\n\n")

        f.write("SHAPE:\n")
        f.write(pformat(group["shape"], width=160))
        f.write("\n\n")

        f.write("ALPHAS:\n")
        f.write(pformat(group["alphas"], width=160))
        f.write("\n\n")

        f.write("REDUCED EDGES:\n")
        f.write(pformat(group["edges"], width=160))
        f.write("\n\n")

        f.write("PATTERNS BY ALPHA:\n")

        for alpha in group["alphas"]:
            f.write(f"  alpha = {alpha}\n")

            for pattern, support in group["patterns_by_alpha"][alpha]:
                f.write(f"    pattern = {pformat(pattern, width=160)}\n")
                f.write(f"    support = {support}\n")

            f.write("\n")

        f.write("\n\n")

print("File scritto: alpha_groups_sorted_nontrivial.txt")

==== GROUP 1 ====
shape_size: 6
alpha_count: 3
pattern_count: 16
edge_count: 2
is_proportional: True
shape: (('22520', 1), ('22521', 1), ('22522', 1), ('22523', 1), ('22524', 1), ('22525', 1))
alphas: [1, 2, 12]
edges: [(1, 2, 2), (2, 12, 6)]

==== GROUP 2 ====
shape_size: 5
alpha_count: 3
pattern_count: 25
edge_count: 2
is_proportional: True
shape: (('22520', 1), ('22521', 1), ('22522', 1), ('22523', 1), ('22524', 1))
alphas: [1, 2, 12]
edges: [(1, 2, 2), (2, 12, 6)]

==== GROUP 3 ====
shape_size: 5
alpha_count: 3
pattern_count: 20
edge_count: 2
is_proportional: True
shape: (('22520', 1), ('22521', 1), ('22523', 1), ('22524', 1), ('22525', 1))
alphas: [1, 2, 12]
edges: [(1, 2, 2), (2, 12, 6)]

==== GROUP 4 ====
shape_size: 5
alpha_count: 3
pattern_count: 19
edge_count: 2
is_proportional: True
shape: (('22520', 1), ('22521', 1), ('22522', 1), ('22524', 1), ('22525', 1))
alphas: [1, 2, 12]
edges: [(1, 2, 2), (2, 12, 6)]

==== GROUP 5 ====
shape_size: 5
alpha_count: 3
pattern_count: 16
e

File scritto: alpha_groups_sorted_nontrivial.txt


In [28]:
def parent_map_to_dot(parent_map, graph_name="G"):
    """
    Convert a dict {node_tuple: parent_tuple_or_None} into Graphviz DOT.

    Example input:
        {
            ("B", 2): ("A", 1),
            ("C", 3): ("A", 1),
            ("A", 1): None,
        }
    """
    lines = [f'digraph "{graph_name}" {{']

    # Collect all nodes
    nodes = set(parent_map.keys())
    nodes.update(parent for parent in parent_map.values() if parent is not None)

    # Stable internal IDs
    node_ids = {node: f"n{i}" for i, node in enumerate(sorted(nodes, key=repr))}

    # Node declarations with tuple labels
    for node in sorted(nodes, key=repr):
        label = repr(node).replace('"', r"\"")
        lines.append(f'    {node_ids[node]} [label="{label}"];')

    # Edges: parent -> child
    for child, parent in parent_map.items():
        if parent is not None:
            lines.append(f'    {node_ids[parent]} -> {node_ids[child]};')

    lines.append("}")
    return "\n".join(lines)

def find_proportion(pattern1: tuple, pattern2: tuple) -> float | None:
    from math import isclose
    
    alpha = None
    for i1, qt1 in pattern1:
        qt1 = abs(qt1)
        for i2, qt2 in pattern2:
            qt2 = abs(qt2)
            if alpha is None and i1 == i2:
                alpha = qt1 / qt2
            elif i1 == i2 and not isclose(qt1 / qt2, alpha):
                return None
    
    return alpha

def find_proportion_roots(patterns: list[tuple]) -> dict[tuple, tuple]:
    pattern_parent = {pattern: None for pattern in patterns}

    # Group by items contained in pattern
    items_map = {}
    for pattern in patterns:
        pattern_items = tuple(sorted(item for item, _ in pattern))
        if pattern_items not in items_map:
            items_map[pattern_items] = set()
        items_map[pattern_items].add(pattern)
    
    for items in items_map:
        ordered_patterns = []
        patterns = items_map[items]
        for pattern in patterns:
            ordered_patterns.append(tuple(sorted(pattern, reverse=True, key=lambda x: (x[1], x[0]))))
        
        items_map[items] = list(sorted(ordered_patterns, reverse=True))

    for _, patterns in items_map.items():
        for p1 in patterns:
            if pattern_parent[p1] is not None:
                continue
            for p2 in patterns:
                if pattern_parent[p2] is not None:
                    continue
                if sorted(p1) == sorted(p2):
                    continue

                alpha = find_proportion(p1, p2)
                if alpha is None:
                    continue
                if alpha < 1:
                    p2, p1 = p1, p2
                pattern_parent[p1] = p2
    
    with open('forest.dot', "w") as f:
        f.write(parent_map_to_dot(pattern_parent))

    return pattern_parent

def groupby_alpha(patterns: list[tuple[tuple, int]]) -> dict[tuple, tuple]:
    no_support_patterns = [pattern for pattern, _ in patterns]
    return find_proportion_roots(no_support_patterns)


real_patterns.mapPartitions(groupby_alpha, True).collect()

26/05/25 21:35:43 ERROR Executor: Exception in task 0.0 in stage 94.0 (TID 41)1]
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/matthewexe/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3386, in main
    process()
  File "/home/matthewexe/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3375, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/matthewexe/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/core/rdd.py", line 705, in func
    return f(iterator)
           ^^^^^^^^^^^
  File "/tmp/ipykernel_45636/1340796780.py", line 92, in groupby_alpha
  File "/tmp/ipykernel_45636/1340796780.py", line 70, in find_proportion_roots
KeyError: (('DOT', 1), ('90000D', 1))

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePytho

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 94.0 failed 1 times, most recent failure: Lost task 0.0 in stage 94.0 (TID 41) (matthewexe.fritz.box executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/matthewexe/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3386, in main
    process()
  File "/home/matthewexe/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3375, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/matthewexe/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/core/rdd.py", line 705, in func
    return f(iterator)
           ^^^^^^^^^^^
  File "/tmp/ipykernel_45636/13407967